In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RAW   = "data/GDSC2_fitted_dose_response_27Oct23.xlsx"
MODEL = "data/model_list_20260619.csv"

In [2]:
def build():
    d = pd.read_excel(RAW)
    m = pd.read_csv(MODEL, low_memory=False)
    m = m[m.model_type == "Cell Line"][["model_id", "tissue"]]
    d = d.merge(m, left_on="SANGER_MODEL_ID", right_on="model_id", how="left")
    d = d.drop(columns=["model_id"]).reset_index(drop=True)
    assert d.tissue.isna().sum() == 0
    return d

d = build()
d.to_parquet("data/gdsc2_clean.parquet")
print(d.shape)

(242036, 17)


In [3]:
def make_splits(df, seed=0):
    s = {}
    idx = np.asarray(df.index)

    tr, te = train_test_split(idx, test_size=0.2, random_state=seed)
    s["random"] = (tr, te)

    cls = np.asarray(df.SANGER_MODEL_ID.unique())
    _, te_cl = train_test_split(cls, test_size=0.2, random_state=seed)
    m = np.asarray(df.SANGER_MODEL_ID.isin(te_cl))
    s["cell_blind"] = (idx[~m], idx[m])

    drs = np.asarray(df.DRUG_ID.unique())
    _, te_dr = train_test_split(drs, test_size=0.2, random_state=seed)
    m = np.asarray(df.DRUG_ID.isin(te_dr))
    s["drug_blind"] = (idx[~m], idx[m])

    return s

In [4]:
splits = make_splits(d)
np.savez("data/splits.npz", **{f"{k}_{p}": v
    for k, (tr, te) in splits.items() for p, v in [("tr", tr), ("te", te)]})

for k, (tr, te) in splits.items():
    print(f"{k:12s} train {len(tr):>7,}  test {len(te):>6,}")

random       train 193,628  test 48,408
cell_blind   train 193,128  test 48,908
drug_blind   train 191,097  test 50,939


In [5]:
def per_drug_r(df, idx, pred):
    pred = pd.Series(np.asarray(pred), index=pd.Index(idx))
    t = df.loc[idx, ["DRUG_ID", "AUC"]].copy()
    assert (t.index == pred.index).all()
    t["pred"] = pred
    rs = {}
    for k, g in t.groupby("DRUG_ID"):
        if len(g) > 4 and g.pred.nunique() > 1 and g.AUC.nunique() > 1:
            rs[k] = g.AUC.corr(g.pred)
    return (np.mean(list(rs.values())) if rs else np.nan), len(rs)

In [6]:
print(f"{'split':12s} {'RMSE':>8s}  {'per-drug r':>12s}  {'drugs':>6s}")
for name, (tr, te) in splits.items():
    mu = d.loc[tr].groupby("DRUG_ID").AUC.mean()
    pred = d.loc[te, "DRUG_ID"].map(mu).fillna(d.loc[tr, "AUC"].mean()).to_numpy()
    rmse = np.sqrt(((d.loc[te, "AUC"].to_numpy() - pred) ** 2).mean())
    r, n = per_drug_r(d, te, pred)
    print(f"{name:12s} {rmse:8.4f}  {str(r):>12s}  {n:>6d}")

split            RMSE    per-drug r   drugs
random         0.0959           nan       0
cell_blind     0.0965           nan       0
drug_blind     0.1370           nan       0
